# Unit Tests — test_train_model.py

Tests every model's train and test functions using small synthetic data.
Run each cell to verify all models work correctly.

In [ ]:
import os
os.chdir('..')
print(os.getcwd())

## Setup

In [14]:
# NOTE: If kagglehub is not installed, run one of these in a Jupyter cell:
#   !pip install kagglehub
#   !pip3 install kagglehub
#   import sys; !{sys.executable} -m pip install kagglehub

import sys
!{sys.executable} -m pip install kagglehub


In [19]:
import numpy as np
import pandas as pd
import sys, os
sys.path.insert(0, '.')

# Import with aliases to avoid name collision with test functions
from src.train_model import (
    train_trivial as _train_trivial,
    test_trivial as _eval_trivial,
    train_ridge as _train_ridge,
    test_ridge as _eval_ridge,
    train_rf as _train_rf,
    test_rf as _eval_rf,
    train_xgboost_default as _train_xgb_default,
    test_xgboost_default as _eval_xgb_default,
    train_xgboost_tuned as _train_xgb_tuned,
    test_xgboost_tuned as _eval_xgb_tuned,
    train_stacking as _train_stacking,
    test_stacking as _eval_stacking,
    run_ablation as _run_ablation,
)
print("All imports successful")

All imports successful


## Create Synthetic Model Data

In [16]:
# Create small synthetic train/val/test data for model testing
def create_model_data():
    np.random.seed(42)
    n_features = 10
    feature_names = [f'feature_{i:02d}' for i in range(n_features)]

    def make_df(n_rows):
        X = pd.DataFrame(np.random.randn(n_rows, n_features), columns=feature_names)
        y = pd.Series(np.random.randn(n_rows), name='responder_6')
        w = pd.Series(np.random.uniform(0.5, 2.0, n_rows), name='weight')
        return X, y, w

    # 1000 train, 200 val, 200 test, small enough to run in seconds
    X_train, y_train, w_train = make_df(1000)
    X_val, y_val, w_val = make_df(200)
    X_test, y_test, w_test = make_df(200)

    return {
        'X_train': X_train, 'y_train': y_train, 'w_train': w_train,
        'X_val': X_val, 'y_val': y_val, 'w_val': w_val,
        'X_test': X_test, 'y_test': y_test, 'w_test': w_test,
        'feature_names': feature_names,
    }

model_data = create_model_data()
print(f"Train: {model_data['X_train'].shape}, Val: {model_data['X_val'].shape}, Test: {model_data['X_test'].shape}")

Train: (1000, 10), Val: (200, 10), Test: (200, 10)


## Test: Trivial Baseline

In [20]:
# Verify trivial baseline trains and produces predictions with R^2 near 0
def test_trivial():
    train_mean, val_rmse, val_r2 = _train_trivial(
        model_data['y_train'], model_data['y_val'], model_data['w_val'])
    test_rmse, test_r2 = _eval_trivial(
        train_mean, model_data['y_test'], model_data['w_test'])
    assert isinstance(train_mean, float), "FAIL: train_mean should be float"
    assert test_rmse > 0, "FAIL: test_rmse should be positive"
    assert abs(val_r2) < 0.1, "FAIL: trivial R^2 should be near 0"
    

test_trivial()

= Trivial Baseline =
Training mean: -0.039909
[Validate] RMSE: 1.0845 | R^2: -0.0112
[Test]     RMSE: 0.9985 | R^2: -0.0004



## Test: Ridge Regression

In [21]:
# Verify Ridge trains and produces correct number of predictions
def test_ridge():
    model, scaler, val_rmse, val_r2 = _train_ridge(
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'])
    preds, test_rmse, test_r2 = _eval_ridge(
        model, scaler, model_data['X_test'], model_data['y_test'], model_data['w_test'])
    assert len(preds) == len(model_data['y_test']), f"FAIL: expected {len(model_data['y_test'])} predictions, got {len(preds)}"
    assert test_rmse > 0, "FAIL: test_rmse should be positive"

test_ridge()

= Ridge Regression ==
[Train]    Time: 0.1s
[Validate] RMSE: 1.0854 | R^2: -0.0129
[Test]     RMSE: 1.0029 | R^2: -0.0092



## Test: Random Forest

In [22]:
# Verify Random Forest trains and produces correct number of predictions
# Using small n_estimators for fast testing
def test_random_forest():
    model, val_rmse, val_r2 = _train_rf(
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'],
        n_estimators=10, max_depth=5, min_samples_leaf=10)
    preds, test_rmse, test_r2 = _eval_rf(
        model, model_data['X_test'], model_data['y_test'], model_data['w_test'])
    assert len(preds) == len(model_data['y_test']), f"FAIL: expected {len(model_data['y_test'])} predictions, got {len(preds)}"
    assert test_rmse > 0, "FAIL: test_rmse should be positive"
    

test_random_forest()

= Random Forest =
[Train]    Time: 0.1s
[Validate] RMSE: 1.0905 | R^2: -0.0224
[Test]     RMSE: 1.0164 | R^2: -0.0366



## Test: XGBoost Default

In [23]:
# Verify XGBoost Default trains and produces correct number of predictions
def test_xgboost_default():
    model, val_rmse, val_r2 = _train_xgb_default(
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'],
        n_estimators=10, max_depth=3)
    preds, test_rmse, test_r2 = _eval_xgb_default(
        model, model_data['X_test'], model_data['y_test'], model_data['w_test'])
    assert len(preds) == len(model_data['y_test']), f"FAIL: expected {len(model_data['y_test'])} predictions, got {len(preds)}"
    assert test_rmse > 0, "FAIL: test_rmse should be positive"
    

test_xgboost_default()

= XGBoost (Default) =
[Train]    Time: 0.0s
[Validate] RMSE: 1.0852 | R^2: -0.0126
[Test]     RMSE: 1.0080 | R^2: -0.0195



## Test: XGBoost Tuned (with early stopping)

In [25]:
# Verify XGBoost Tuned trains with early stopping kicking in before max iterations
def test_xgboost_tuned():
    model, val_rmse, val_r2 = _train_xgb_tuned(
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'],
        n_estimators=500, max_depth=3, early_stopping_rounds=10)
    preds, test_rmse, test_r2 = _eval_xgb_tuned(
        model, model_data['X_test'], model_data['y_test'], model_data['w_test'])
    assert len(preds) == len(model_data['y_test']), f"FAIL: expected {len(model_data['y_test'])} predictions, got {len(preds)}"
    assert model.best_iteration < 500, f"FAIL: early stopping should kick in before 500, got {model.best_iteration}"
    

test_xgboost_tuned()

= XGBoost (Tuned) =
Best iteration: 18
[Train]    Time: 0.1s
[Validate] RMSE: 1.0791 | R^2: -0.0012
[Test]     RMSE: 0.9983 | R^2: -0.0001



## Test: Stacking Ensemble

In [26]:
# Verify Stacking trains base models, blends via meta-model with 2 coefficients
def test_stacking():
    # Train base models first
    ridge_model, ridge_scaler, _, _ = _train_ridge(
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'])
    xgbt_model, _, _ = _train_xgb_tuned(
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'],
        n_estimators=50, max_depth=3, early_stopping_rounds=10)
    # Train and test stacking
    meta_model, meta_scaler, val_rmse, val_r2 = _train_stacking(
        ridge_model, ridge_scaler, xgbt_model,
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'])
    preds, test_rmse, test_r2 = _eval_stacking(
        meta_model, meta_scaler, ridge_model, ridge_scaler, xgbt_model,
        model_data['X_test'], model_data['y_test'], model_data['w_test'])
    assert len(preds) == len(model_data['y_test']), f"FAIL: expected {len(model_data['y_test'])} predictions, got {len(preds)}"
    assert len(meta_model.coef_) == 2, f"FAIL: meta-model should have 2 coefficients, got {len(meta_model.coef_)}"
    

test_stacking()

= Ridge Regression ==
[Train]    Time: 0.0s
[Validate] RMSE: 1.0854 | R^2: -0.0129
= XGBoost (Tuned) =
Best iteration: 18
[Train]    Time: 0.1s
[Validate] RMSE: 1.0791 | R^2: -0.0012
=== Stacking Ensemble (Ridge + XGBoost Tuned -> Meta Ridge) ===
[Train]    Time: 0.0s
[Validate] RMSE: 1.0581 | R^2: 0.0373
  Meta-model coefficients: Ridge=-0.0324, XGBoost=0.2185
[Test]   RMSE: 1.0158 | R^2: -0.0353



## Test: Ablation Study

In [27]:
# Verify ablation returns DataFrame with correct structure and number of rows
def test_ablation():
    feature_groups = {
        'Group A': model_data['feature_names'][:5],
        'Group B': model_data['feature_names'],
    }
    result = _run_ablation(
        feature_groups,
        model_data['X_train'], model_data['y_train'], model_data['w_train'],
        model_data['X_val'], model_data['y_val'], model_data['w_val'],
        model_data['X_test'], model_data['y_test'], model_data['w_test'])
    assert len(result) == 2, f"FAIL: expected 2 rows, got {len(result)}"
    assert 'Test RMSE' in result.columns, "FAIL: Test RMSE column missing"

    print(result.to_string(index=False))

test_ablation()

 Ablation: Group A (5 features) ===
[Validate] RMSE: 1.0846
[Test]     RMSE: 0.9939 | R^2: 0.0087
Time: 0.2s

 Ablation: Group B (10 features) ===
[Validate] RMSE: 1.0796
[Test]     RMSE: 0.9952 | R^2: 0.0062
Time: 0.2s

Feature Set  Num Features  Val RMSE  Test RMSE  Test R^2
    Group A             5  1.084572   0.993944  0.008680
    Group B            10  1.079613   0.995188  0.006197
